# 6.15 ROC-AUC Curve

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 6: Classification Algorithms

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-06-classification-algorithms/06.15_roc-auc-curve.ipynb)

Companion code for Section 6.15 of the book. Run the cells in order from top to bottom.

**Example 6.15.1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
import xgboost as xgb

data = load_breast_cancer()
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

# ── Compare multiple classifiers on ROC ───────────────────────────
classifiers = {
    'Logistic Regression':  LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':        RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':    GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)':            SVC(kernel='rbf', probability=True, random_state=42),
    'Naive Bayes':          GaussianNB(),
    'XGBoost':              xgb.XGBClassifier(n_estimators=100, eval_metric='logloss',
                                               use_label_encoder=False, random_state=42, verbosity=0),
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('ROC-AUC Analysis - Multiple Classifiers', fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot([0,1],[0,1],'k--',lw=1.5,alpha=0.7,label='Random (AUC=0.5)')

print(f'{'Model':<22} {'AUC':>8} {'Accuracy':>10}')
print('='*45)

for name, clf in classifiers.items():
    clf.fit(X_tr_s, y_tr)
    y_prob = clf.predict_proba(X_te_s)[:,1]
    fpr, tpr, thresholds = roc_curve(y_te, y_prob)
    roc_auc = auc(fpr, tpr)
    acc = clf.score(X_te_s, y_te)
    ax.plot(fpr, tpr, lw=2.5, label=f'{name} (AUC={roc_auc:.4f})')
    print(f'{name:<22} {roc_auc:>8.4f} {acc:>10.4f}')

ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
ax.set_title('ROC Curves - All Classifiers')
ax.legend(loc='lower right', fontsize=9); ax.grid(alpha=0.3)

# ── Cross-validated ROC with confidence band ──────────────────────
ax = axes[1]
rf = RandomForestClassifier(n_estimators=100, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
mean_fpr = np.linspace(0, 1, 100)
tprs, aucs = [], []

for i, (train, test) in enumerate(cv.split(X_tr_s, y_tr)):
    rf.fit(X_tr_s[train], y_tr[train])
    y_prob_cv = rf.predict_proba(X_tr_s[test])[:,1]
    fpr_cv, tpr_cv, _ = roc_curve(y_tr[test], y_prob_cv)
    interp_tpr = np.interp(mean_fpr, fpr_cv, tpr_cv)
    interp_tpr[0] = 0.0
    tprs.append(interp_tpr)
    aucs.append(auc(fpr_cv, tpr_cv))
    ax.plot(fpr_cv, tpr_cv, alpha=0.2, lw=1, color='#1976D2')

mean_tpr = np.mean(tprs, axis=0); mean_tpr[-1] = 1.0
std_tpr  = np.std(tprs, axis=0)
mean_auc = auc(mean_fpr, mean_tpr)

ax.plot(mean_fpr, mean_tpr, 'b-', lw=3,
       label=f'Mean ROC (AUC={mean_auc:.4f} ± {np.std(aucs):.4f})')
ax.fill_between(mean_fpr, mean_tpr-std_tpr, mean_tpr+std_tpr,
               alpha=0.15, color='blue', label='±1 std dev')
ax.plot([0,1],[0,1],'k--',lw=1.5,alpha=0.7,label='Random')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('5-Fold CV ROC Curve\n(Random Forest with confidence band)')
ax.legend(loc='lower right', fontsize=10); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('roc_auc_curves.png', dpi=150, bbox_inches='tight')
plt.show()